In [ ]:
library(Seurat)
library(MuDataSeurat)
library(CellChat)
library(patchwork)
library(reticulate)
library(uwot)
library(ComplexHeatmap)
library(wordcloud)

# Read in RDS objects

In [ ]:
# Load the cellchat objects from RDS file with centrality scores
#cellchat  <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_centr_sub.rds")

cellchat.ctrl <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_ctrl_centr_sub.rds")
cellchat.age <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_age_centr_sub.rds")
cellchat.DR <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_DR_centr_sub.rds")
cellchat.sAct <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_sAct_centr_sub.rds")
cellchat.combi <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_interorgan/cellchat_objects/cellchat_combi_centr_sub.rds")

In [ ]:
levels(cellchat.ctrl@idents)
levels(cellchat.age@idents)
levels(cellchat.DR@idents)
levels(cellchat.sAct@idents)
levels(cellchat.combi@idents)

## Merge Objects

In [ ]:
# Lift cellchat objects for merge
group.new = union(levels(cellchat.ctrl@idents), levels(cellchat.ctrl@idents))
cellchat.ctrl <- liftCellChat(cellchat.ctrl, group.new)

# Merge to one object
object.list <- list(ctrl = cellchat.ctrl, age = cellchat.age, DR = cellchat.DR, sAct = cellchat.sAct, combi = cellchat.combi)# Create a named list of cellchat objects

cellchat <- mergeCellChat(object.list, add.names = names(object.list), cell.prefix = TRUE)
cellchat <- updateCellChat(cellchat)

In [ ]:
levels(cellchat@idents)

levels(cellchat@meta[['supertype']])
levels(cellchat@meta[['subtype']])
levels(cellchat@meta[['celltype']])
levels(cellchat@meta[['mixed_celltype']])

## Subset merged object

In [ ]:
# Define the cell types you want to include
desired_cell_types <- c("Myonuclei", "Endothelial cells", "EC-1(gEC)", "PODO", "PT-1", "PT-2", "vSMC/MC", "'PECs?", "IMM")

# Subset the cellchat object based on desired cell types
sub_cellchat <- subsetCellChat(cellchat, idents.use = desired_cell_types)

sort(unique(cellchat@idents$joint))
sort(unique(sub_cellchat@idents$joint))

# netVisual_circle

In [ ]:
# Loop through each cellchat object and apply visualization code
for(i in 1:length(object.list)){
          
          cellchat_name <- names(object.list)[i]
          cellchat_obj <- object.list[[i]]
          groupSize <- as.numeric(table(object.list[[i]]@idents)) # number of cells in each cell group
          
          par(mfrow = c(1,2), xpd=TRUE)
          #par(mfrow = c(1,1), xpd=TRUE)
          netVisual_circle(cellchat_obj@net$count, vertex.weight = groupSize, weight.scale = TRUE, label.edge = FALSE, title.name = paste("Number of interactions -", cellchat_name))
          netVisual_circle(cellchat_obj@net$weight, vertex.weight = groupSize, weight.scale = TRUE, label.edge = FALSE, title.name = paste("Weight of interactions -", cellchat_name))
      }

In [ ]:
#sub_cellchat@idents$ctrl
#sub_cellchat@net$ctrl

# Comparison analysis of multiple datasets with different cell type compositions

In [ ]:
pathway_names <- CellChatDB.mouse$interaction['pathway_name']
head(pathway_names)
unique(CellChatDB.mouse$interaction$pathway_name)

## Part I: Comparison analysis of multiple datasets with slightly different cell type compositions

In [ ]:
cellchat_objects <- list(
    list(object = cellchat.ctrl, name = "ctrl"),
    list(object = cellchat.age, name = "aging"),
    list(object = cellchat.DR, name = "DR"),
    list(object = cellchat.sAct, name = "sAct"),
    list(object = cellchat.combi, name = "combi")
)

# Extract cell type annotations from each object
celltype_annotations <- lapply(cellchat_objects, function(item) {
  cellchat_obj <- item$object
  cellchat_name <- item$name
  celltypes <- unique(cellchat_obj@idents)
  cat("Cell types in", cellchat_name, ":", paste(celltypes, collapse = ", "), "\n")
  return(celltypes)
})

# Compare cell type compositions
for (i in 1:(length(cellchat_objects) - 1)) {
  for (j in (i + 1):length(cellchat_objects)) {
    obj1_name <- cellchat_objects[[i]]$name
    obj2_name <- cellchat_objects[[j]]$name
    
    if (!identical(celltype_annotations[[i]], celltype_annotations[[j]])) {
      cat("Cell type compositions are different between", obj1_name, "and", obj2_name, "\n")
    } else {
      cat("Cell type compositions are the same between", obj1_name, "and", obj2_name, "\n")
    }
  }
}

### Visualize the inferred signaling network & specific pathways

In [ ]:
# Find common pathways to display
all_pathways <- list()

# Iterate through each named object and extract pathways
for (object_name in names(object.list)) {
   # print(object_name)
  cellchat_obj <- object.list[[object_name]]
  pathways <- cellchat_obj@netP$pathways
  all_pathways[[object_name]] <- pathways
}

# Print the list of combined pathways
all_pathways

# Extract the pathways that are present in all objects
common_pathways <- Reduce(intersect, all_pathways)
common_pathways

In [ ]:
par(mfrow = c(2,3), xpd=TRUE)
pathway = "PTPRM"

for(i in 1:length(object.list)){
    cellchat_name <- names(object.list)[i]
    cellchat_obj <- object.list[[i]]
    groupSize <- as.numeric(table(object.list[[i]]@idents)) # number of cells in each cell group
    
    par(mfrow=c(1,1))
    netVisual_aggregate(cellchat_obj , signaling = pathway, signaling.name = paste(pathway, names(object.list)[i]))
    netVisual_aggregate(cellchat_obj, signaling =pathway, layout = "chord", signaling.name = paste(pathway, names(object.list)[i]))
    heat <- netVisual_heatmap(cellchat_obj, signaling = pathway, color.heatmap = "Reds", measure= 'weight',title.name = paste(pathway, names(object.list)[i]))
    print(heat)
}

In [ ]:
# Visualize all pathways
for (pathways_list in common_pathways) {
  for (pathway in pathways_list) {
      par(mfrow = c(2,3), xpd=TRUE)
      for(i in 1:length(object.list)){
          cellchat_name <- names(object.list)[i]
          cellchat_obj <- object.list[[i]]
          groupSize <- as.numeric(table(object.list[[i]]@idents)) # number of cells in each cell group
          
          netVisual_aggregate(cellchat_obj , signaling = pathway, signaling.name = paste(pathway, names(object.list)[i]))
        #  netVisual_heatmap(cellchat_obj, signaling = pathway, color.heatmap = "Reds", measure= 'weight', title.name = paste(pathway, names(object.list)[i])
      }
  }
}

In [ ]:
for(i in 1:length(object.list)){
    cellchat_name <- names(object.list)[i]
    cellchat_obj <- object.list[[i]]
   
    # control the parameter edge.weight.max so that we can compare edge weights between differet networks.
    weight_mat <- cellchat_obj@net$weight
    
   # png(filename = "celltype_signal.png", bg = "transparent", width = 2000, height = 1250)
    #par(mfrow = c(3, 3),mgp=c(0,0,0), xpd=TRUE)
    
    png(filename = paste("./figures/", as.character(paste("celltype_signal_", cellchat_name)), ".png", sep = ""), bg = "white", width = 1500, height = 1500)
    par(mfrow = c(3, 3),mgp=c(0,0,0), xpd=TRUE)
    
    for (cel in unique(cellchat_obj@idents)){
        
        
        cir_mat <- matrix(0, nrow = nrow(weight_mat), ncol = ncol(weight_mat), dimnames = dimnames(weight_mat))
        cir_mat[cel, ] <- weight_mat[cel, ]
        
       # options(repr.plot.width = 25, repr.plot.height = 15)  # Set the width and height as per your requirement
        netVisual_circle( cir_mat, vertex.weight= groupSize, weight.scale= T, edge.weight.max = max(5 * weight_mat), vertex.label.cex=0.8, title.name=paste(cel, "-", cellchat_name))
       
    }
     dev.off()
}

# Comparison analysis of multiple datasets using CellChat

## Differential number of interactions or interaction strength among different cell populations

The differential number of interactions or interaction strength in the cell-cell communication network between two datasets can be visualized using circle plot, where red (or blue) colored edges represent increased (or decreased) signaling in the second dataset compared to the first one.

In [ ]:
# Set the size of the plot
options(repr.plot.width=6, repr.plot.height=6)
par(mfrow = c(1,2), xpd=TRUE)

netVisual_diffInteraction(cellchat, comparison = c(1, 2), weight.scale = T, measure = "weight")
netVisual_diffInteraction(cellchat, comparison = c(1, 2), weight.scale = T, measure = "count")

We can also show differential number of interactions or interaction strength in a greater details using a heatmap. The top colored bar plot represents the sum of column of values displayed in the heatmap (incoming signaling). The right colored bar plot represents the sum of row of values (outgoing signaling). In the colorbar, red (or blue) represents increased (or decreased) signaling in the second dataset compared to the first one.

In [ ]:
for (object_name in names(object.list)) {
  cellchat_obj <- object.list[[object_name]]
    
    # Set the size of the plot
    options(repr.plot.width=15, repr.plot.height=10)
    
    gg1 <- netVisual_heatmap(cellchat_obj, title.name = paste("Number of interactions in ", object_name))
    #> Do heatmap based on a merged object
    gg2 <- netVisual_heatmap(cellchat_obj, measure = "weight", title.name = paste("Interaction strength in", object_name))
    #> Do heatmap based on a merged object
    print(gg1 + gg2)
}


In [ ]:
weight.max <- getMaxWeight(object.list, attribute = c("idents","count"))
par(mfrow = c(1,2), xpd=TRUE)
for (i in 1:length(object.list)) {
  netVisual_circle(object.list[[i]]@net$count, weight.scale = T, label.edge= F, edge.weight.max = weight.max[2], edge.width.max = 12, title.name = paste0("Number of interactions - ", names(object.list)[i]))
}

## Part I: Predict general principles of cell-cell communication

In [ ]:
levels(cellchat@meta$sample)
head(cellchat@meta)

In [ ]:
# Compare the total number of interactions and interaction strength
plot_title <- "All celltypes"

gg1 <- compareInteractions(cellchat, show.legend = F, group = c(1,2,3,4,5), title.name = plot_title)
gg2 <- compareInteractions(cellchat, show.legend = F, group = c(1,2,3,4,5), measure = "weight")
gg1 + gg2

...

In [ ]:
# Create a vector of sample names to compare with 'ctrl'
samples_to_compare <- c('age', 'DR','sAct', 'combi')

# Initialize an empty list to store the heatmaps
heatmap_list <- list()

# Loop through each sample to compare with 'ctrl'
for (sample in samples_to_compare) {
  # Create the comparison vector
  comparison_vector <- c('ctrl', sample)
  
  # Generate the heatmaps
    gg1 <- netVisual_heatmap(cellchat, comparison = comparison_vector, title.name = paste("Nb interactions:", comparison_vector[1], "vs", comparison_vector[2]), width = 3,  height = 5)
    gg2 <- netVisual_heatmap(cellchat, comparison = comparison_vector, measure = "weight", title.name = paste("Interaction strength:", comparison_vector[1], "vs", comparison_vector[2]))
   # print(gg1 + gg2)
    print(gg1)
    # Define the file path for saving the heatmap (change the format to 'png' or 'pdf' as needed)
    file_path <- file.path("./figures", paste0("heatmap_", sample, ".png"))
    
    #ggsave(file_path, plot = gg1 , width = 10, height = 6, units = "in")
}

In [ ]:
# Create a vector of sample names to compare with 'ctrl'
samples_to_compare <- c('ctrl', 'DR','sAct', 'combi')

# Initialize an empty list to store the heatmaps
heatmap_list <- list()

# Loop through each sample to compare with 'ctrl'
for (sample in samples_to_compare) {
  # Create the comparison vector
  comparison_vector <- c('age', sample)
  
  # Generate the heatmaps
    gg1 <- netVisual_heatmap(cellchat, comparison = comparison_vector, title.name = paste("Nb interactions:", comparison_vector[1], "vs", comparison_vector[2]))
    gg2 <- netVisual_heatmap(cellchat, comparison = comparison_vector, measure = "weight", title.name = paste("Interaction strength:", comparison_vector[1], "vs", comparison_vector[2]))
    #print(gg1 + gg2)
    print(gg1)
    
    # Define the file path for saving the heatmap (change the format to 'png' or 'pdf' as needed)
    file_path <- file.path("./cellchat_figures", paste0("heatmap_", sample, ".pdf"))
}

...

### Compare the major sources and targets in 2D space

'Comparing the outgoing and incoming interaction strength in 2D space allows ready identification of the cell populations with significant changes in sending or receiving signals between different datasets.

In [ ]:
ptm = Sys.time()
# Compute the network centrality scores
#cellchat <- netAnalysis_computeCentrality(cellchat, slot.name = "netP") # the slot 'netP' means the inferred intercellular communication network of signaling pathways

cellchat.ctrl <- netAnalysis_computeCentrality(cellchat.ctrl, slot.name = "netP")
cellchat.age <- netAnalysis_computeCentrality(cellchat.age, slot.name = "netP")
cellchat.DR <- netAnalysis_computeCentrality(cellchat.DR, slot.name = "netP")
cellchat.sAct <- netAnalysis_computeCentrality(cellchat.sAct, slot.name = "netP")
cellchat.combi <- netAnalysis_computeCentrality(cellchat.combi, slot.name = "netP")

In [ ]:
#pathways.show = c("PECAM1", "NCAM", "JAM","ANGPT","THBS", "CADM", "CD200", "NOTCH", "PROS")
pathways.show = c("NOTCH", "PROS")
for (i in 1:length(object.list)) {
    cellchat_name <- names(object.list)[i]
    cellchat_obj <- object.list[[i]]
    
    print(cellchat_name)
    netAnalysis_signalingRole_network(cellchat_obj, signaling = pathways.show, width = 12, height = 5, font.size = 10)
}


In [ ]:
num.link <- sapply(object.list, function(x) {rowSums(x@net$count) + colSums(x@net$count)-diag(x@net$count)})
num.link <- unlist(num.link)
weight.MinMax <- c(min(num.link), max(num.link)) # control the dot size in the different datasets
gg <- list()



for (i in 1:length(object.list)) {
    png(filename = paste("./figures/", as.character( paste("signal_", names(object.list)[i])), ".png", sep = ""), bg = "white", width = 1250, height = 1250)
    par(mfrow = c(2, 3),mgp=c(0,0,0), xpd=TRUE)
    
    gg[[i]] <- netAnalysis_signalingRole_scatter(object.list[[i]],  title = names(object.list)[i],  weight.MinMax = weight.MinMax) +  scale_y_continuous(limits = c(0,20)) + scale_x_continuous(limits = c(0,15))
    print(gg[[i]])
    dev.off()
}


widths <- c(15, 15, 15)  # Adjust these values
heights <- c(15, 15)    # Adjust these values

patchwork::wrap_plots(plots = gg, widths = widths, heights = heights)

In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(2, 3, 4, 5)

# Initialize an empty list to store the plots
gg <- list()

# Loop through each sample to compare with ctrl
for (i in 1:length(desired_cell_types)) {
    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        gg[[i]] <- netAnalysis_signalingChanges_scatter(cellchat, idents.use = desired_cell_types[i], comparison = comparison_vector)
        print(gg[[i]])
       # patchwork::wrap_plots(plots = list(gg[1], gg[2]))
    }
}

## Part II: Identify the conserved and context-specific signaling pathways

#### New netEmbedding function

In [ ]:
netEmbedding <- function(object, slot.name = "netP", type = c("functional","structural"), comparison = NULL, pathway.remove = NULL, k = NULL) {
  
    if (object@options$mode == "single") {
    comparison <- "single"
    cat("Manifold learning of the signaling networks for a single dataset", '\n')
  } else if (object@options$mode == "merged") {
    if (is.null(comparison)) {
      comparison <- 1:length(unique(object@meta$datasets))
    }
    cat("Manifold learning of the signaling networks for datasets", as.character(comparison), '\n')
  }
  comparison.name <- paste(comparison, collapse = "-")
  Similarity <- methods::slot(object, slot.name)$similarity[[type]]$matrix[[comparison.name]]
  if (is.null(pathway.remove)) {
    pathway.remove <- rownames(Similarity)[which(colSums(Similarity) == 1)]
  }
  if (length(pathway.remove) > 0) {
    pathway.remove.idx <- which(rownames(Similarity) %in% pathway.remove)
    Similarity <- Similarity[-pathway.remove.idx, -pathway.remove.idx]
  }
  if (is.null(k)) {
    k <- ceiling(sqrt(dim(Similarity)[1])) + 1
  }
  options(warn = -1)
  # dimension reduction
  Y <- umap(Similarity, min_dist = 0.3, n_neighbors = k)
  if (!is.list(methods::slot(object, slot.name)$similarity[[type]]$dr)) {
    methods::slot(object, slot.name)$similarity[[type]]$dr <- NULL
  }
  methods::slot(object, slot.name)$similarity[[type]]$dr[[comparison.name]] <- Y
  return(object)
}

### Identify signaling groups based on their functional similarity

In [ ]:
ptm = Sys.time()

cellchat <- computeNetSimilarityPairwise(cellchat, type = "functional")
cellchat <- netEmbedding(cellchat, type = "functional")
cellchat <- netClustering(cellchat, type = "functional")

# Visualization in 2D-space
netVisual_embeddingPairwise(cellchat, type = "functional", label.size = 3.5)
netVisual_embeddingPairwiseZoomIn(cellchat, type = "functional", nCol = 2)

### Identify signaling groups based on structure similarity

In [ ]:
cellchat <- computeNetSimilarityPairwise(cellchat, type = "structural")
cellchat <- netEmbedding(cellchat, type = "structural")
cellchat <- netClustering(cellchat, type = "structural")

# Visualization in 2D-space
netVisual_embeddingPairwise(cellchat, type = "structural", label.size = 3.5)
netVisual_embeddingPairwiseZoomIn(cellchat, type = "structural", nCol = 2)

### Compute and visualize the pathway distance in the learned joint manifold

In [ ]:
#rankSimilarity(cellchat, type = "functional")

### Identify and visualize the conserved and context-specific signaling pathways

#### Compare the overall information flow of each signaling pathway

In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(2, 3, 4, 5)

# Loop through each sample to compare with ctrl
    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        gg1 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = T, do.stat = TRUE)
        gg2 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = F, do.stat = TRUE)
        
        print(gg1 + gg2)
    }


In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(1,  3, 4, 5)

# Loop through each sample to compare with ctrl

    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(2, sample)
        # Generate the plots
        gg1 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = T, do.stat = TRUE)
        gg2 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = F, do.stat = TRUE)
        
        print(gg1 + gg2)
    }

#### Compare outgoing (or incoming) signaling associated with each cell population

In [ ]:
i = 1
# combining all the identified signaling pathways from different datasets 
pathway.union <- union(object.list[[i]]@netP$pathways, object.list[[i+1]]@netP$pathways)
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i], width = 15, height =20)
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+1], width = 15, height =20)
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+2], width = 15, height =20)
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+3], width = 15, height =20)
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+4], width = 15, height =20)

#draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
#draw(ht3 + ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)


png(file="./figures/combined_heatmap.png",width=100,height=200)
#combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
#draw(combined_heatmap)
dev.off()

In [ ]:
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i], width = 15, height =20, color.heatmap = "GnBu")
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+1], width = 15, height =20, color.heatmap = "GnBu")
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+2], width = 15, height =20, color.heatmap = "GnBu")
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+3], width = 15, height =20, color.heatmap = "GnBu")
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+4], width = 15, height =20, color.heatmap = "GnBu")

#draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
#draw(ht3+ ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)


png(file="./figures/combined_heatmap.png",width=100,height=200)
#combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
#draw(combined_heatmap)
dev.off()

In [ ]:
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "all", signaling = pathway.union, title = names(object.list)[i], width = 15, height =20, color.heatmap = "OrRd")
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+1], width = 15, height =20, color.heatmap = "OrRd")
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+2], width = 15, height =20, color.heatmap = "OrRd")
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+3], width = 15, height =20, color.heatmap = "OrRd")
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+4], width = 15, height =20, color.heatmap = "OrRd")

#draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
#draw(ht3+ ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)

png(file="./figures/combined_heatmap.png",width=100,height=200)
#combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
#draw(combined_heatmap)
dev.off()

## Part III: Identify the upgulated and down-regulated signaling ligand-receptor pairs

### Identify dysfunctional signaling by comparing the communication probabities

In [ ]:
# Assuming 'comparison' is a vector of dataset numbers
comparison <- c(1, 2,3, 4, 5)

# Get the dataset names from the 'sub_cellchat' object
dataset_names <- names(cellchat@images)

# Extract the dataset names corresponding to the numbers in 'comparison'
selected_dataset_names <- dataset_names[comparison]

# Print the selected dataset names
cat("Selected dataset names:", selected_dataset_names, "\n")

In [ ]:
options(repr.plot.width = 60, repr.plot.height = 15)  
netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, 2, 3, 4, 5), angle.x = 90)

In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(2, 3, 4, 5)

# Loop through each sample to compare with ctrl

    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        options(repr.plot.width = 20, repr.plot.height = 10)  
        # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
        net1 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, sample), max.dataset = sample, title.name = paste("Increased signaling in", names(object.list)[sample], "(vs. ctrl)"), angle.x = 90, remove.isolate = T)
        #> Comparing communications on a merged object
        net2 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, sample), max.dataset = 1, title.name = paste("Decreased signaling in ", names(object.list)[sample], "(vs. ctrl)"), angle.x = 90, remove.isolate = T)
        print(net1)
        print(net2)
    }

In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(1, 3, 4, 5)

# Loop through each sample to compare with ctrl

for (sample in samples_to_compare) {
    # Create the comparison vector
    comparison_vector <- c(2, sample)
    # Generate the plots
    options(repr.plot.width = 20, repr.plot.height = 20)  
    # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
    net1 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = comparison_vector, max.dataset = sample, title.name = paste("Increased signaling in", names(object.list)[sample], "(vs. age)"), angle.x = 90, remove.isolate = T)
    #> Comparing communications on a merged object
    net2 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = comparison_vector, max.dataset = 2, title.name = paste("Decreased signaling in ", names(object.list)[sample], "(vs. age)"), angle.x = 90, remove.isolate = T)
    print(net1)
    print(net2)
}

### Identify dysfunctional signaling by using differential expression analysis

In [ ]:
# Create a vector of sample names to compare with ctrl
samples_to_compare <- c(2, 3, 4, 5)

# define a positive dataset, i.e., the dataset with positive fold change against the other dataset
pos.dataset = "ctrl"
# define a char name used for storing the results of differential expression analysis
features.name = pos.dataset
# perform differential expression analysis
cellchat <- identifyOverExpressedGenes(cellchat, group.dataset = "datasets", pos.dataset = pos.dataset, features.name = features.name, only.pos = FALSE, thresh.pc = 0.1, thresh.fc = 0.1, thresh.p = 1)

# map the results of differential expression analysis onto the inferred cell-cell communications to easily manage/subset the ligand-receptor pairs of interest
net <- netMappingDEG(cellchat, features.name = features.name)
# extract the ligand-receptor pairs with upregulated ligands in ctrl

for (sample in samples_to_compare) {
    # Create the comparison vector
    comparison_vector <- c(1, sample)
    # Generate the plots
    options(repr.plot.width = 20, repr.plot.height = 10)  
    
    # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
    net.up <- subsetCommunication(cellchat, net = net, datasets ="ctrl",ligand.logFC = 0.2, receptor.logFC = NULL)
    # extract the ligand-receptor pairs with upregulated ligands and upregulated recetptors in sample2, i.e.,downregulated in ctrl
    net.down <- subsetCommunication(cellchat, net = net, datasets = names(object.list)[sample],ligand.logFC = -0.1, receptor.logFC = -0.1)
    
    gene.up <- extractGeneSubsetFromPair(net.up, cellchat)
    gene.down <- extractGeneSubsetFromPair(net.down, cellchat)
    
    # We then visualize the upgulated and down-regulated signaling ligand-receptor pairs using bubble plot
    options(repr.plot.width = 20, repr.plot.height = 15)
    pairLR.use.up = net.up[, "interaction_name", drop = F]
    gg1 <- netVisual_bubble(cellchat, pairLR.use = pairLR.use.up, sources.use = NULL, targets.use = NULL, comparison = comparison_vector,  angle.x = 90, remove.isolate = T,title.name = paste0("Up-regulated signaling in ", names(object.list)[1], " vs.", names(object.list)[sample]))
    pairLR.use.down = net.down[, "interaction_name", drop = F]
    gg2 <- netVisual_bubble(cellchat, pairLR.use = pairLR.use.down, sources.use = NULL, targets.use = NULL, comparison = comparison_vector,  angle.x = 90, remove.isolate = T,title.name = paste0("Down-regulated signaling in ", names(object.list)[1], " vs.", names(object.list)[sample]))

    print(gg1)
    print(gg2)
    
    #or visualize the upgulated and down-regulated signaling ligand-receptor pairs using Chord diagram
    netVisual_chord_gene(object.list[[sample]], sources.use = NULL, targets.use = NULL, slot.name = 'net', net = net.up, lab.cex = 0.8, small.gap = 3.5, title.name = paste0("Up-regulated signaling in ", names(object.list)[sample]))
    netVisual_chord_gene(object.list[[1]], sources.use = NULL, targets.use = NULL, slot.name = 'net', net = net.down, lab.cex = 0.8, small.gap = 3.5, title.name = paste0("Down-regulated signaling in ", names(object.list)[sample]))
    
    # visualize the enriched ligands in the first condition
computeEnrichmentScore(net.down, species = 'mouse')
    # visualize the enriched ligands in the second condition
computeEnrichmentScore(net.up, species = 'mouse')
   }

# New plots

## Identify global communication patterns to explore how multiple cell types and signaling pathways coordinate together

### Identify and visualize outgoing communication pattern of secreting cells

In [ ]:
library(NMF)
library(ggalluvial)

#### Outgoing

In [ ]:
selectK(cellchat, pattern = "outgoing")

selectK(cellchat.ctrl, pattern = "outgoing")
selectK(cellchat.age, pattern = "outgoing")
selectK(cellchat.DR, pattern = "outgoing")
selectK(cellchat.sAct, pattern = "outgoing")
selectK(cellchat.combi, pattern = "outgoing")

In [ ]:
#### Full cellchat object

In [ ]:
# Both Cophenetic and Silhouette values begin to drop suddenly when the number of outgoing patterns is xxx
nPatterns = 3

# Outgoing patterns reveal how the sender cells (i.e. cells as signal source) coordinate with each other as well as how they coordinate with certain signaling pathways to drive communication.
cellchat <- identifyCommunicationPatterns(cellchat, pattern = "outgoing", k = nPatterns, width = 15, height = 23)
# river plot
netAnalysis_river(cellchat, pattern = "outgoing")
netAnalysis_dot(cellchat, pattern = "outgoing")

In [ ]:
#### Separated cellchat objects

In [ ]:
i = 5

names(object.list)[i]
identifyCommunicationPatterns(object.list[[i]], pattern = "outgoing", k = nPatterns, width = 15, height = 23, heatmap.show = TRUE)

In [ ]:
# Both Cophenetic and Silhouette values begin to drop suddenly when the number of outgoing patterns is xxx
nPatterns = 3



for (i in 1:length(object.list)) {
    cellchat_name <- names(object.list)[i]
    cellchat_obj <- object.list[[i]]
    
    # Outgoing patterns reveal how the sender cells (i.e. cells as signal source) coordinate with each other as well as how they coordinate with certain signaling pathways to drive communication.
    cellchat_obj <- identifyCommunicationPatterns(cellchat_obj, pattern = "outgoing", k = nPatterns, width = 15, height = 23, heatmap.show = TRUE)
   
    png(filename = paste("./figures/", as.character(paste("patterns_out_river_", cellchat_name)), ".png", sep = ""), bg = "white", width = 1250, height = 1250)
    par(mfrow = c(1, 1),mgp=c(0,0,0), xpd=TRUE)
    
    # river plot
    river <- netAnalysis_river(cellchat_obj, pattern = "outgoing", main.title = paste("Outgoing communication patterns -", cellchat_name), font.size = 5, font.size.title = 18)
    netAnalysis_river(cellchat_obj, pattern = "outgoing", main.title = paste("Outgoing communication patterns -", cellchat_name), font.size = 5, font.size.title = 18)
    dot <- netAnalysis_dot(cellchat_obj, pattern = "outgoing", main.title = paste("Outgoing communication patterns -", cellchat_name), font.size = 15)
   
    print(river)
    #print(dot)
    
    dev.off()
}

#### Incoming

In [ ]:
#### Full cellchat object

In [ ]:
# Both Cophenetic and Silhouette values begin to drop suddenly when the number of outgoing patterns is xxx
nPatterns = 3

# incoming patterns reveal how the sender cells (i.e. cells as signal source) coordinate with each other as well as how they coordinate with certain signaling pathways to drive communication.
cellchat <- identifyCommunicationPatterns(cellchat, pattern = "incoming", k = nPatterns, width = 15, height = 23)

# river plot
netAnalysis_river(cellchat, pattern = "incoming")
netAnalysis_dot(cellchat, pattern = "incoming")

In [ ]:
#### Separated cellchat objects

In [ ]:
i = 5

names(object.list)[i]
identifyCommunicationPatterns(object.list[[i]], pattern = "incoming", k = nPatterns, width = 15, height = 23, heatmap.show = TRUE)

In [ ]:
# Both Cophenetic and Silhouette values begin to drop suddenly when the number of outgoing patterns is xxx
nPatterns = 3

for (i in 1:length(object.list)) {
    
    cellchat_name <- names(object.list)[i]
    cellchat_obj <- object.list[[i]]
    

# Outgoing patterns reveal how the sender cells (i.e. cells as signal source) coordinate with each other as well as how they coordinate with certain signaling pathways to drive communication.
    cellchat_obj <- identifyCommunicationPatterns(cellchat_obj, pattern = "incoming", k = nPatterns, width = 15, height = 23, heatmap.show = FALSE)
    
    png(filename = paste("./figures/", as.character(paste("patterns_in_dot_", cellchat_name)), ".png", sep = ""), bg = "white", width = 1250, height = 1250)
    par(mfrow = c(1, 1),mgp=c(0,0,0), xpd=TRUE)
    # river plot
   # river <- netAnalysis_river(cellchat_obj, pattern = "incoming", main.title = paste("Incoming communication patterns -", cellchat_name), font.size = 5, font.size.title = 18)
     #netAnalysis_river(cellchat_obj, pattern = "incoming", main.title = paste("Incoming communication patterns -", cellchat_name), font.size = 5, font.size.title = 18)
    dot <- netAnalysis_dot(cellchat_obj, pattern = "incoming", main.title = paste("Incoming communication patterns -", cellchat_name), font.size = 15)
    netAnalysis_dot(cellchat_obj, pattern = "incoming", main.title = paste("Incoming communication patterns -", cellchat_name), font.size = 15)
    
    #print(river)
    print(dot)
    
    dev.off()
    
}

## Manifold and classification learning analysis of signaling networks

CellChat is able to quantify the similarity between all significant signaling pathways and then group them based on their cellular communication network similarity. Grouping can be done either based on the functional or structural similarity.

Functional similarity: High degree of functional similarity indicates major senders and receivers are similar, and it can be interpreted as the two signaling pathways or two ligand-receptor pairs exhibit similar and/or redundant roles. The functional similarity analysis requires the same cell population composition between two datasets.
Structural similarity: A structural similarity was used to compare their signaling network structure, without considering the similarity of senders and receivers.


In [ ]:
cellchat <- computeNetSimilarity(cellchat, type = "structural")
cellchat <- netEmbedding(cellchat, type = "structural")
#> Manifold learning of the signaling networks for a single dataset
cellchat <- netClustering(cellchat, type = "structural")
netVisual_embeddingZoomIn(cellchat, type = "structural", nCol = 2)

In [ ]:
# Save Cellchat object

# Export

In [ ]:
# Export interorgan CellChat LR interactions per condition
# Adjust the object name to match your CellChat workflow

for (cond_name in names(object.list)) {
  df <- subsetCommunication(object.list[[cond_name]])
  write.csv(df, paste0("interorgan_", cond_name, "_lr_interactions.csv"), 
            row.names = FALSE)
  cat(cond_name, ":", nrow(df), "interactions\n")
}